In [14]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
import warnings

import CM4Xutils
import numpy as np
import pandas as pd
import xarray as xr
import xgcm
import xhistogram

import gsw, xwmt
import zarr

In [16]:
import cmocean
import matplotlib.colors as colors
import matplotlib.ticker as mtick
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from mpl_toolkits.axes_grid1 import make_axes_locatable
plt.rcParams.update({'font.size': 12})

In [17]:
xgcm.__version__, xwmt.__version__

('0.9.0', '0.1.0')

### Load data and create metadata structures

In [18]:
from common import *
grids = load_datasets()
sim = 'CM4Xp125_forced'
grid = grids[sim]
ds = grid._ds

Inferring Z grid coordinate: depth `z_`
Inferring Z grid coordinate: depth `z_`


### Compute outcrop frequencies

In [6]:
coords = {
    "start_year": xr.DataArray([1990, 2090], dims=("start_year",)),
    "end_year": xr.DataArray([1999, 2099], dims=("start_year",)),
    "rho2_moc_l": moc_metrics.rho2_moc_l,
    "rho2_moc_i": moc_metrics.rho2_moc_i,
    "sigma2_moc_l": moc_metrics.sigma2_moc_l,
    "sigma2_moc_i": moc_metrics.sigma2_moc_i,
}
ds_outcrops = xr.Dataset(coords=coords)
ds_outcrops["outcrop_frequency"] = xr.zeros_like(xr.broadcast(ds_outcrops.start_year, ds_outcrops.rho2_moc_l, ds.areacello)[0])

for start_year in ds_outcrops.start_year:
    years = [str(start_year.values), str(ds_outcrops.end_year.sel(start_year=start_year).values)]
    sigma = ds.sigma2_surface.sel(time=slice(*years))
    
    for k in range(moc_metrics.rho2_moc_i.size - 1):
        layer_outcrop_mask = (
            (ds_outcrops.sigma2_moc_i.isel(rho2_moc_i=k) < sigma) &
            (sigma <= ds_outcrops.sigma2_moc_i.isel(rho2_moc_i=k+1))
        )
        ds_outcrops["outcrop_frequency"] = xr.where(
            (ds_outcrops.start_year == start_year) & (ds_outcrops.rho2_moc_l==ds_outcrops.rho2_moc_l.isel(rho2_moc_l=k)),
            layer_outcrop_mask.sum("time") / layer_outcrop_mask.time.size * 100,
            ds_outcrops["outcrop_frequency"]
        )

ds_outcrops.to_netcdf(f"../data/processed/layer_outcrop_frequencies_{sim}.nc", mode="w")

### Layer-wise inventory

In [7]:
ds["volcello"] = ds.thkcello.fillna(0.)*ds.areacello.fillna(0.)
ds["cfc11_content"] = ds.cfc11.fillna(0.)*ds.volcello
ds["sigma2_nanfilled"] = CM4Xutils.fillna_below(grid, ds.sigma2).transpose("year", "z_l", "yh", "xh")

ds["sigma2_i"] = grid.interp(
    ds["sigma2_nanfilled"],
    "Z",
    boundary="extend"
)

In [8]:
outcrop_area_by_layer = xhistogram.xarray.histogram(
    ds.sigma2_surface,
    bins=moc_metrics.sigma2_moc_i.values,
    dim=("xh", "yh",),
    weights=ds.areacello,
    bin_dim_suffix="_l",
).groupby("time.year").mean("time").rename({"sigma2_surface_l": "rho2_moc_l"})

In [9]:
thickness_by_layer = grid.transform(
    ds.thkcello.fillna(0.),
    "Z",
    target=moc_metrics.sigma2_moc_i,
    target_data=ds.sigma2_i,
    method="conservative",
).rename({"rho2_moc_i":"rho2_moc_l"})

In [10]:
volcello_by_layer = grid.transform(
    ds.volcello,
    "Z",
    target=moc_metrics.sigma2_moc_i,
    target_data=ds.sigma2_i,
    method="conservative"
).rename({"rho2_moc_i":"rho2_moc_l"})

In [11]:
cfc11_content_by_layer = grid.transform(
    ds.cfc11_content,
    "Z",
    target=moc_metrics.sigma2_moc_i,
    target_data=ds.sigma2_i,
    method="conservative",
).rename({"rho2_moc_i":"rho2_moc_l"})

In [12]:
inv_layer = xr.Dataset()
inv_layer["outcrop_area"] = outcrop_area_by_layer.compute()
inv_layer["volume"] = volcello_by_layer.sum(["xh", "yh"]).compute()
inv_layer["cfc11_content"] = cfc11_content_by_layer.sum(["xh", "yh"]).compute()

In [13]:
inv_layer.to_netcdf(f"../data/processed/layer_cfc11_inventories_{sim}.nc", mode="w")